# HeyTea CNN Training — Kaggle Notebook

**Before running:**
1. Upload `heytea_images.zip` (รูป 5 คลาส) เป็น Kaggle Dataset ชื่อ **`heytea-images`**
2. Add that dataset to this notebook (+ Add Input → Your Datasets)
3. Enable GPU: Settings → Accelerator → **GPU T4 x2** (or P100)
4. Enable Internet: Settings → Internet → On (ใช้ clone repo + โหลด pretrained weights)
5. (ไม่บังคับ) W&B: Add-ons → Secrets → `WANDB_API_KEY` — ไม่ใส่ก็รันได้ (ปิด logging)

**metadata_clean.csv + split.csv อ่านจาก repo ที่ clone (git)** — Kaggle dataset มีแค่รูป
→ แก้ split แล้วแค่ push ขึ้น git ไม่ต้องอัปรูปใหม่

**Run:** กด **Save Version → Save & Run All (Commit)** — รันเบื้องหลังได้ถึง 12 ชม. ปิดเบราว์เซอร์ได้
ผลอยู่ในแท็บ Output: `results/runs.csv`, `summary.csv`, `ttest.csv`, รูปกราฟ
ถ้าเวลาไม่พอ: ตั้ง `ARCHS_TO_RUN` ทีละ 1-2 arch ต่อ version แล้วรวม `runs.csv` ทีหลัง

**Runs ที่ทำแล้ว / แผน**
- Version 2: Option B (final, default hyperparameters) ✅
- Version 3: Option C (imbalance ablation)
- Version 4: Option D (Optuna + W&B Sweep → final ใหม่ด้วยค่าที่ tune) ~6–7 ชม. บน T4
  ก่อนรัน: ใส่ secret `WANDB_API_KEY` (Add-ons → Secrets) ไม่งั้นจะรันแค่ Optuna


In [ ]:
# ── Install extra packages ────────────────────────────────────────────────────
# torch/torchvision already available on Kaggle
!pip install -q wandb scikit-learn scipy

In [ ]:
# ── Clone repo for training code + metadata + split ──────────────────────────
import subprocess
!rm -rf /kaggle/working/repo
result = subprocess.run(
    ['git', 'clone', '--depth', '1', 'https://github.com/tanpim16/dl_dads7202_heytea', '/kaggle/working/repo'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
!cd /kaggle/working/repo && git log --oneline -1


In [ ]:
# ── Override paths for Kaggle filesystem ─────────────────────────────────────
import sys
from pathlib import Path

sys.path.insert(0, '/kaggle/working/repo/training')

import config
# หาโฟลเดอร์รูปอัตโนมัติ (path ของ Kaggle dataset ต่างกันได้ตามวิธีอัป zip)
hits = sorted(Path('/kaggle/input').glob('**/cha_dam_yen'))
assert hits, 'ไม่เจอโฟลเดอร์รูป — เพิ่ม dataset heytea-images เข้า notebook หรือยัง?'
config.DATA_DIR       = hits[0].parent
REPO_DATA             = Path('/kaggle/working/repo/tea_dataset')
config.METADATA       = REPO_DATA / 'metadata_clean.csv'   # จาก git
config.SPLIT_FILE     = REPO_DATA / 'split.csv'            # จาก git
config.CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
config.RESULTS_DIR    = Path('/kaggle/working/results')
config.NUM_WORKERS    = 2

config.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
meta = pd.read_csv(config.METADATA)
missing = [r for r in meta['relpath'] if not (config.DATA_DIR / r).exists()]
print('DATA_DIR:', config.DATA_DIR)
print(f'images in metadata: {len(meta)} | missing on Kaggle: {len(missing)}', missing[:5])
assert not missing, 'รูปใน metadata_clean.csv ไม่ครบใน Kaggle dataset -> อัป zip ใหม่'
print('IMBALANCE:', config.IMBALANCE)


In [ ]:
# ── (Optional) Override BEST_HPARAMS after sweep ─────────────────────────────
# Fill these in from the W&B sweep results before final run
config.BEST_HPARAMS = {
    'vgg16': {
        'stage1_lr': 0.001,  'stage1_epochs': 10,
        'stage2_lr': 0.0001, 'stage2_epochs': 15,
        'label_smoothing': 0.05,
    },
    'resnet50': {
        'stage1_lr': 0.001,  'stage1_epochs': 10,
        'stage2_lr': 0.0001, 'stage2_epochs': 15,
        'label_smoothing': 0.0,
    },
    'efficientnet_b3': {
        'stage1_lr': 0.001,  'stage1_epochs': 10,
        'stage2_lr': 0.0001, 'stage2_epochs': 15,
        'label_smoothing': 0.0,
    },
    'mobilenet_v3_large': {
        'stage1_lr': 0.001,  'stage1_epochs': 5,
        'stage2_lr': 0.00005,'stage2_epochs': 20,
        'label_smoothing': 0.0,
    },
}

In [ ]:
# ── W&B login (ไม่บังคับ) ───────────────────────────────────────────────────────
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    import wandb; wandb.login()
    print('W&B logged in')
except Exception as e:
    os.environ.pop('WANDB_API_KEY', None)
    print('W&B ปิด (ไม่มี secret WANDB_API_KEY) — รันต่อได้ปกติ')


In [ ]:
# ── Verify split + data ───────────────────────────────────────────────────────
from dataset import make_splits, make_loaders, get_class_weights

train_df, val_df, test_df = make_splits()
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
print('\nTrain class counts:')
print(train_df.groupby('class').size().to_string())
print('\nTest class counts:')
print(test_df.groupby('class').size().to_string())

In [ ]:
# ── Option A: (เลิกใช้) — tuning ย้ายไป Option D (Optuna + W&B Sweep แยกต่อ arch) ──
RUN_SWEEP = False   # คงตัวแปรไว้ให้ cell อื่นอ้างถึง ห้ามตั้ง True


In [ ]:
# ── Option B: Final 5-seed run — 4 architectures ────────────────────────────
# ผลแต่ละ run เขียนลง results/runs.csv ทันที; run ที่มีผลแล้วจะถูกข้าม
RUN_FINAL     = True
ARCHS_TO_RUN  = None   # None = ทั้ง 4 arch | หรือเช่น ['resnet50', 'mobilenet_v3_large']

if RUN_FINAL and not RUN_SWEEP:
    from run_final import run_all
    run_all(ARCHS_TO_RUN)
    print('\nDone! Results saved to /kaggle/working/results/')


In [ ]:
# ── Option C: Class-imbalance experiment (none / class_weight / sampler / focal) ──
# 1 arch × 5 seeds × 4 วิธี -> results/imbalance_runs.csv, imbalance_summary_*.csv, imbalance_ttest_*.csv
# รอบที่ 2 บน Kaggle: ตั้ง RUN_FINAL = False (cell บน) และ RUN_IMBALANCE = True
RUN_IMBALANCE   = False
IMBALANCE_ARCH  = 'resnet50'

if RUN_IMBALANCE:
    import run_imbalance as R
    R.run(IMBALANCE_ARCH, R.METHODS)
    R.summarize()


In [ ]:
# ── Option D: Hyperparameter tuning — Optuna + W&B Sweep, แล้วรัน final ใหม่ด้วยค่าที่ดีที่สุด ──
# 1) Optuna (TPE + median pruning) ทุก arch          -> results/tuning/trials.csv
# 2) W&B Sweep (Bayesian) ทุก arch (ต้องมี WANDB_API_KEY) -> trials.csv (tool = wandb_sweep) + หน้า W&B
# 3) เลือกค่าที่ดีที่สุดต่อ arch (val F1, ไม่แตะ test)   -> results/tuning/best_hparams.json
# 4) run_final 5 seeds ด้วยค่านั้น                     -> results_tuned/ (ผลรอบ default ไม่ถูกทับ)
# ใช้: ตั้ง RUN_FINAL = False (Option B), RUN_IMBALANCE = False (Option C), RUN_TUNING = True
RUN_TUNING          = False
OPTUNA_TRIALS       = 10     # ต่อ arch
SWEEP_COUNT         = 10     # ต่อ arch (ข้ามอัตโนมัติถ้าไม่มี WANDB_API_KEY)
RUN_FINAL_TUNED     = True   # รัน run_final ต่อทันทีหลัง tuning

if RUN_TUNING:
    import os
    import run_tune_optuna, run_sweep, tuning
    run_tune_optuna.main(None, OPTUNA_TRIALS)
    if os.environ.get('WANDB_API_KEY'):
        run_sweep.main(None, SWEEP_COUNT)
    else:
        print('ข้าม W&B Sweep: ไม่มี WANDB_API_KEY')
    best_json = tuning.tuning_dir() / 'best_hparams.json'
    tuning.select_best(best_json)
    if RUN_FINAL_TUNED:
        from run_final import run_all
        run_all(None, hparams=best_json, out='/kaggle/working/results_tuned')


In [ ]:
# ── Show saved result files ───────────────────────────────────────────────────
import os
results_dir = Path('/kaggle/working/results')
for f in sorted(results_dir.iterdir()):
    print(f.name, f'({f.stat().st_size // 1024} KB)')